In [ ]:
from pyspark.sql import functions as F

# Catalog is a job parameter (default -> ${var.catalog}); the default keeps interactive runs working.
dbutils.widgets.text("catalog", "transport_vic_dev")
CATALOG = dbutils.widgets.get("catalog")

BRONZE_TBL = f"{CATALOG}.`01_bronze`.stops"
SILVER_TBL = f"{CATALOG}.`02_silver`.stops"
CHECKPOINT = f"/Volumes/{CATALOG}/02_silver/_checkpoints/stops"

spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.`02_silver`._checkpoints")

# SCD2 identity = (feed_id, stop_id): stop_id is unique within a feed, but repeats across the mode
# feeds (1..11), so feed_id is part of the key.
BUSINESS_KEY = ["feed_id", "stop_id"]

# The nine ORIGINAL stops.txt attributes. Any change here opens a new version; the pipeline's own
# columns (_source_file, _ingest_ts, export_date) are deliberately excluded from change detection.
TRACKED = ["stop_name", "stop_lat", "stop_lon", "stop_url", "location_type",
           "parent_station", "wheelchair_boarding", "level_id", "platform_code"]

In [ ]:
def project(df):
    """Bronze raw-strings -> silver typed row. lat/lon cast to double; the rest stay as strings
    (they're codes/ids, cast later if a query needs them). export_date -> DATE drives validity."""
    return df.select(
        F.col("feed_id"),
        F.col("stop_id"),
        F.col("stop_name"),
        F.col("stop_lat").try_cast("double").alias("stop_lat"),   # hash over the typed value, so
        F.col("stop_lon").try_cast("double").alias("stop_lon"),   # "-37.8" vs "-37.80" is NOT a change
        F.col("stop_url"),
        F.col("location_type"),
        F.col("parent_station"),
        F.col("wheelchair_boarding"),
        F.col("level_id"),
        F.col("platform_code"),
        F.to_date(F.col("export_date"), "yyyyMMdd").alias("export_date"),
        F.col("_source_file"),
    )

def add_hash(df):
    # xxhash64 is null-safe over its argument list, so feeds missing platform_code (null) don't
    # collide or throw. Order is fixed by TRACKED, so the hash is stable run-to-run.
    return df.withColumn("_row_hash", F.xxhash64(*[F.col(c) for c in TRACKED]))

In [ ]:
# MERGE needs the target to exist. Create it empty with the SCD2 columns; first microbatch then
# seeds every stop as version 1 (valid_from = its export_date, valid_to = null, is_current = true).
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_TBL} (
  feed_id STRING, stop_id STRING,
  stop_name STRING, stop_lat DOUBLE, stop_lon DOUBLE, stop_url STRING,
  location_type STRING, parent_station STRING, wheelchair_boarding STRING,
  level_id STRING, platform_code STRING,
  _row_hash BIGINT,
  valid_from DATE, valid_to DATE, is_current BOOLEAN,
  _source_file STRING, _silver_ingest_ts TIMESTAMP
) USING DELTA
""")

In [ ]:
VAL_COLS = ["feed_id", "stop_id", "stop_name", "stop_lat", "stop_lon", "stop_url",
            "location_type", "parent_station", "wheelchair_boarding", "level_id",
            "platform_code", "_row_hash", "_source_file"]

def _merge_one_export(slice_df, export_date):
    """Apply one export's snapshot as an SCD2 step: close changed current rows, insert new versions."""
    current = (spark.read.table(SILVER_TBL)
                    .filter("is_current = true")
                    .select(*BUSINESS_KEY, F.col("_row_hash").alias("_cur_hash")))

    joined = slice_df.join(current, BUSINESS_KEY, "left")
    # New key (no current row) OR attributes changed (hash differs) -> a version must be inserted.
    to_change = joined.filter(F.col("_cur_hash").isNull() | (F.col("_row_hash") != F.col("_cur_hash")))
    if to_change.isEmpty():
        return  # this export changed nothing -> no-op (the common weekly case)

    # Canonical Delta SCD2 stage: null mergeKey rows INSERT a new version; real mergeKey rows MATCH
    # the existing current row so it can be closed. A changed key appears in BOTH.
    insert_stage = to_change.drop("_cur_hash").withColumn("_mergeKey", F.lit(None).cast("string"))
    close_stage  = (to_change.filter(F.col("_cur_hash").isNotNull())
                             .drop("_cur_hash")
                             .withColumn("_mergeKey", F.concat_ws("~", *BUSINESS_KEY)))

    (insert_stage.unionByName(close_stage)
                 .withColumn("valid_from", F.lit(export_date).cast("date"))
                 .createOrReplaceTempView("_stops_scd2_stage"))

    spark.sql(f"""
      MERGE INTO {SILVER_TBL} t
      USING _stops_scd2_stage s
        ON concat_ws('~', t.feed_id, t.stop_id) = s._mergeKey AND t.is_current = true
      WHEN MATCHED THEN UPDATE SET
        t.is_current = false,
        t.valid_to   = s.valid_from
      WHEN NOT MATCHED THEN INSERT (
        feed_id, stop_id, stop_name, stop_lat, stop_lon, stop_url,
        location_type, parent_station, wheelchair_boarding, level_id, platform_code,
        _row_hash, valid_from, valid_to, is_current, _source_file, _silver_ingest_ts
      ) VALUES (
        s.feed_id, s.stop_id, s.stop_name, s.stop_lat, s.stop_lon, s.stop_url,
        s.location_type, s.parent_station, s.wheelchair_boarding, s.level_id, s.platform_code,
        s._row_hash, s.valid_from, cast(null as date), true, s._source_file, current_timestamp()
      )
    """)

def upsert_scd2(batch_df, batch_id):
    if batch_df.isEmpty():
        return
    typed = add_hash(project(batch_df))
    # A single microbatch can span many export_dates (esp. the first/backfill run). Apply them in
    # date order so history is built in sequence; each iteration re-reads silver, seeing the prior one.
    export_dates = [r["export_date"] for r in
                    typed.select("export_date").distinct().orderBy("export_date").collect()]
    for ed in export_dates:
        slice_df = typed.filter(F.col("export_date") == F.lit(ed)).dropDuplicates(BUSINESS_KEY)
        _merge_one_export(slice_df, ed)

In [ ]:
# Streaming read of bronze -> foreachBatch SCD2 MERGE. The checkpoint is what makes each run process
# only newly-appended bronze rows; availableNow drains what's there and stops (matches bronze + the
# table_update trigger on the silver job). Structured Streaming needs a real serverless job, not Connect.
q = (spark.readStream
          .table(BRONZE_TBL)
          .writeStream
          .option("checkpointLocation", CHECKPOINT)
          .foreachBatch(upsert_scd2)
          .trigger(availableNow=True)
          .start())
q.awaitTermination()   # so the job task doesn't report success before the batch finishes